In [6]:
from pystac_client import Client
import xarray as xr

# 1. Connect and search the STAC catalog
catalog = Client.open("https://stac.itslive.cloud")
search = catalog.search(
    bbox=[88.15, 27.90, 88.25, 27.97],  # South Lhonak bounding box
    collections=["itslive-datacube-v2"],
)

items = list(search.items())

if not items:
    # Fallback in case the collection ID is omitted
    search = catalog.search(bbox=[88.15, 27.90, 88.25, 27.97])
    items = list(search.items())

# Define 'item'
item = items[0]

# 2. Extract and format the Zarr asset URL
zarr_asset = item.assets.get("data") or item.assets.get("zarr")
zarr_url = zarr_asset.href

if "its-live-data.s3.amazonaws.com" in zarr_url:
    zarr_url = zarr_url.replace("https://its-live-data.s3.amazonaws.com", "s3://its-live-data")

# 3. Open the dataset with consolidated metadata and region options
ds = xr.open_dataset(
    zarr_url,
    engine="zarr",
    consolidated=True,
    chunks={},
    storage_options={
        "anon": True,
        "client_kwargs": {"region_name": "us-west-2"}
    }
)

# 4. Filter time and select velocity
ds_subset = ds.sel(time=slice("2017-01-01", "2023-12-31"))

if "v" in ds_subset:
    velocity_magnitude = ds_subset.v
else:
    velocity_magnitude = (ds_subset.vx**2 + ds_subset.vy**2)**0.5

print(velocity_magnitude)

KeyboardInterrupt: 